# 05 - Conversion Prediction Modeling

In this notebook, we build and evaluate machine learning models (Linear Regression, Decision Tree, Random Forest) to predict **Conversions**.

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [2]:
# Load Dataset
df = pd.read_csv(r"C:\Users\ASUS\Downloads\DAML dataset\tech_advertising_campaigns_dataset.csv")

print("Dataset shape:", df.shape)

Dataset shape: (10000, 41)


In [3]:
# Target Variable Summary
print("Conversions summary statistics:")
print(df["conversions"].describe())

Conversions summary statistics:
count    10000.000000
mean       124.520000
std        245.890000
min          0.000000
25%         15.000000
50%         52.000000
75%        142.000000
max       8500.000000
Name: conversions, dtype: float64


In [4]:
df_model = df.drop(columns=["campaign_id"]).copy()

# Convert start_date to datetime
df_model["start_date"] = pd.to_datetime(df_model["start_date"])

# Create date-based features
df_model["start_year"] = df_model["start_date"].dt.year
df_model["start_month"] = df_model["start_date"].dt.month
df_model["start_day"] = df_model["start_date"].dt.day

# Remove original date column
df_model = df_model.drop(columns=["start_date"])

# Convert Boolean features to 0/1
boolean_features = [
    "has_call_to_action",
    "retargeting_flag"
]

for column in boolean_features:
    df_model[column] = df_model[column].astype(int)

print("Prepared dataset shape:", df_model.shape)

Prepared dataset shape: (10000, 43)


In [5]:
# Separate Features and Conversion Target (prevent target leakage)
leakage_columns = [
    "conversions",
    "conversion_rate",
    "CPA",
    "revenue",
    "profit",
    "ROAS"
]
if "ROI" in df_model.columns:
    leakage_columns.append("ROI")

X = df_model.drop(columns=[col for col in leakage_columns if col in df_model.columns])
y = df_model["conversions"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (10000, 37)
y shape: (10000,)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (8000, 37)
X_test: (2000, 37)
y_train: (8000,)
y_test: (2000,)


In [7]:
# Load Saved Preprocessor
preprocessor = joblib.load("../models/preprocessor.pkl")
print("Preprocessor loaded successfully.")

Preprocessor loaded successfully.


In [8]:
# Transform Features
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Models Evaluation
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
}

results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train_processed, y_train)
    y_pred = model.predict(X_test_processed)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    results.append({'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2})
    trained_models[name] = model

results_df = pd.DataFrame(results).set_index('Model')
print("Conversion Model Performance Comparison:")
print(results_df)

Conversion Model Performance Comparison:
                        MAE    RMSE      R2
Model                                      
Linear Regression     85.40  190.20  0.4120
Decision Tree         48.30  145.10  0.6410
Random Forest         36.20  112.50  0.7850


In [9]:
# Save Best Conversion Model
joblib.dump(trained_models['Random Forest'], "../models/conversion_model.pkl")
print("Best Conversion Model (Random Forest) saved successfully to ../models/conversion_model.pkl")

Best Conversion Model (Random Forest) saved successfully to ../models/conversion_model.pkl
